# exp_retriever_bakeoff — is a stronger retriever the lever? (develop on 2021, test on 2023)

The diagnosis: on 2023, retrieval *order* is the result (reranking hurts), and our dense retriever is
weak — 0.32, beaten by BM25 (0.43); IELAB's dense hits 0.576. This notebook swaps in candidate
retrievers and measures **retrieval-order NDCG@10** on the same doc sets, cheaply (encode only the
judged/pool docs, not the full corpus).

**Method (anti-gaming):** rank retrievers on **TREC21** (develop) first; the one that wins there is the
one to trust on **TREC23** (test). Off-the-shelf retrievers (no tuning), so measuring both is fair.

Candidates: `retriever-v2` (current baseline), `BGE-large` (strong general), `MedCPT` (NCBI biomedical,
**asymmetric** query/article encoders — built for the topic≠document mismatch). If a candidate clearly
beats retriever-v2 on 2021 *and* lifts 2023 toward IELAB's 0.576, we do a full-corpus re-retrieval with it.

Requires: the 2021 corpus (`load_corpus`) + the cached `eval_external_2023` 2023 files.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers transformers accelerate datasets pytrec_eval tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
import numpy as np, torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, retrieval_blob, resolve_ckpt, pytrec_metrics
DATA_ROOT = '/content/drive/MyDrive/ct_data23'; T23 = f'{DATA_ROOT}/trec2023'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── TREC23 (test) — cached ──
id2f23 = {}
for l in open(f'{T23}/doc_fulltext_2023.jsonl'):
    r = json.loads(l); id2f23[r.get('nct_id') or r.get('doc_id')] = r
topics23 = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(f'{T23}/topics2023_text.jsonl'))}
rel23 = {}
for l in open(f'{T23}/qrels2023.txt'):
    t, _, d, r = l.split(); rel23.setdefault(t, {})[d] = int(r)
topics23 = {t: x for t, x in topics23.items() if t in rel23}
pool23 = json.load(open(f'{T23}/pool_nqs_2023.json'))

# ── TREC21 (develop, judged pool) ──
corpus_ids, corpus_fields = load_corpus(cfg); id2f21 = dict(zip(corpus_ids, corpus_fields))
s21 = load_eval(cfg, ['trec21'])['trec21']; rel21 = s21['rel_dict']; t2t21 = s21['topic2text']
topics21 = {t: t2t21[t] for t in rel21 if t in t2t21}

DATASETS = {
 'TREC21 (develop, judged pool)': (topics21, lambda t: list(rel21[t]), id2f21,
                                   {t: {d: int(r) for d, r in rel21[t].items()} for t in topics21}),
 'TREC23 (test, NQS pool)':       (topics23, lambda t: pool23[t], id2f23,
                                   {t: {d: int(r) for d, r in rel23[t].items()} for t in topics23}),
}
print('TREC21 topics', len(topics21), '| TREC23 topics', len(topics23))

In [ ]:
# Retriever wrappers — each returns (encode_queries, encode_docs, free). Scoring uses raw dot product,
# so retrievers that normalize give cosine (BGE / retriever-v2) and MedCPT gives its trained dot similarity.
def st_retriever(name, q_prefix='', maxlen=512):
    m = SentenceTransformer(resolve_ckpt(cfg, name)); m.max_seq_length = maxlen
    box = {'m': m}
    def eq(texts): return box['m'].encode([q_prefix + t for t in texts], normalize_embeddings=True,
                                          batch_size=128, show_progress_bar=True).astype('float32')
    def ed(texts): return box['m'].encode(texts, normalize_embeddings=True,
                                          batch_size=128, show_progress_bar=True).astype('float32')
    def free(): box.clear(); torch.cuda.empty_cache()
    return eq, ed, free

def medcpt_retriever():
    box = {'qt': AutoTokenizer.from_pretrained('ncbi/MedCPT-Query-Encoder'),
           'qm': AutoModel.from_pretrained('ncbi/MedCPT-Query-Encoder').eval().to(device),
           'dt': AutoTokenizer.from_pretrained('ncbi/MedCPT-Article-Encoder'),
           'dm': AutoModel.from_pretrained('ncbi/MedCPT-Article-Encoder').eval().to(device)}
    @torch.no_grad()
    def _enc(tk, mk, texts, maxlen, batch=64):
        out = []
        for i in range(0, len(texts), batch):
            e = box[tk](texts[i:i+batch], truncation=True, padding=True, max_length=maxlen, return_tensors='pt').to(device)
            out.append(box[mk](**e).last_hidden_state[:, 0, :].cpu().numpy())   # CLS
        return np.vstack(out).astype('float32')
    def eq(texts): return _enc('qt', 'qm', texts, 64)    # MedCPT query encoder: short-query model (64 tok)
    def ed(texts): return _enc('dt', 'dm', texts, 512)
    def free(): box.clear(); torch.cuda.empty_cache()
    return eq, ed, free

RETRIEVERS = {
    'retriever-v2 (current)': lambda: st_retriever(cfg.retriever_ckpt, maxlen=cfg.retriever_max_tokens),
    'BGE-large':              lambda: st_retriever('BAAI/bge-large-en-v1.5',
                                                   q_prefix='Represent this sentence for searching relevant passages: '),
    'MedCPT (biomedical)':    medcpt_retriever,
}
print('retrievers:', list(RETRIEVERS))

In [ ]:
# Encode topics + the (small) doc set per dataset, rank each topic's docs by similarity, score.
def eval_retriever(eq, ed):
    res = {}
    for dsname, (topics_d, docset_fn, id2f, qrels) in DATASETS.items():
        tids = list(topics_d)
        qv = eq([topics_d[t] for t in tids])
        uniq = sorted({d for t in tids for d in docset_fn(t) if d in id2f})
        dv = ed([retrieval_blob(id2f[d], cfg) for d in uniq]); didx = {d: i for i, d in enumerate(uniq)}
        run = {t: {d: float(qv[k] @ dv[didx[d]]) for d in docset_fn(t) if d in didx} for k, t in enumerate(tids)}
        res[dsname] = pytrec_metrics(run, qrels, k=10)
    return res

results = {}
for name, factory in RETRIEVERS.items():
    print(f'\n=== {name} ===')
    eq, ed, free = factory()
    results[name] = eval_retriever(eq, ed)
    for ds, m in results[name].items(): print(f'  {ds:32s} {m}')
    free()

In [ ]:
# Summary — retrieval-order NDCG@10 (rank on TREC21, read across to TREC23).
print(f'{"retriever":26s} {"TREC21 NDCG@10":>15s} {"TREC23 NDCG@10":>15s}')
print('-'*60)
for name, r in results.items():
    n21 = r['TREC21 (develop, judged pool)']['ndcg@10']; n23 = r['TREC23 (test, NQS pool)']['ndcg@10']
    print(f'{name:26s} {n21:>15.4f} {n23:>15.4f}')
print('-'*60)
print('reference (TREC23): our rrf 0.472 | ensemble 0.399 | monoT5 0.180 | IELAB dense 0.576 | oracle 1.000')
print()
best = max(results, key=lambda n: results[n]['TREC21 (develop, judged pool)']['ndcg@10'])
print(f'2021 winner (the one to trust): {best}  ->  its TREC23 = {results[best]["TREC23 (test, NQS pool)"]["ndcg@10"]:.4f}')
print('If the 2021 winner also clearly lifts TREC23 toward 0.576, do a full-corpus re-retrieval with it')
print('(new pool -> re-extract features -> re-fit ensemble), developed on 2021 and applied once.')